In [7]:
# === Notebook path bootstrap ===
from pathlib import Path
import runpy

CWD = Path.cwd().resolve()
BOOTSTRAP_CANDIDATES = [
    CWD / "land_atmosphere_bootstrap.py",
    CWD / "diagnostic" / "jupyter" / "land_atmosphere_bootstrap.py",
    *[p / "diagnostic" / "jupyter" / "land_atmosphere_bootstrap.py" for p in CWD.parents],
]
BOOTSTRAP_PATH = next((p for p in BOOTSTRAP_CANDIDATES if p.exists()), None)
if BOOTSTRAP_PATH is None:
    raise FileNotFoundError("Could not locate land_atmosphere_bootstrap.py")
globals().update(runpy.run_path(str(BOOTSTRAP_PATH)))

import os
import numpy as np
import warnings 
from string import Template
import xarray as xr
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from scipy.stats import bootstrap
from scipy.stats import linregress

from datetime import datetime

from util.land_atmosphere_observation_data import ( 
     ObsDataReader,
    
)
from util.land_atmosphere_model_data import (
    ModelDataReader,
)

from util.land_atmosphere_experiment_configs import get_experiment_dict


In [8]:
class TerrestrialCouplingIndexCalculator:
    def __init__(self, data_dir, experiment, ensemble_list, years,
                 sm_var='SOILWATER_10CM', flux_var='LHFLX',
                 target_depth_m=None,
                 lat_bounds=None,
                 lon_bounds=None,
                 outdir='./tci_output',
                 landmask_file=None,
                 soilayer_file=None):
        self.data_dir = data_dir
        self.experiment = experiment
        self.ensemble_list = ensemble_list
        self.years = years
        self.sm_var = sm_var
        self.flux_var = flux_var
        self.lat_bounds = lat_bounds
        self.lon_bounds = lon_bounds
        self.outdir = os.path.join(outdir, experiment)
        os.makedirs(self.outdir, exist_ok=True)

        self.landmask = None
        if landmask_file and os.path.exists(landmask_file):
            self.landmask = xr.open_dataset(landmask_file)['landmask']
            self.landmask = self.normalize_longitude_and_sort(self.landmask)
            
        if soilayer_file and os.path.exists(soilayer_file):
            self.DZSOI = xr.open_dataset(soilayer_file)['DZSOI']
        else:
            self.DZSOI = None 
            
        self.Lv = 2.5e6  # J/kg
        self.expected_frequency = 'daily'

        self.regions = {
            "CONUS": {"lat": (25, 50), "lon": (-125, -65)},
            "SouthAmerica": {"lat": (-55, 15), "lon": (-80, -30)},
            "Australia": {"lat": (-45, -10), "lon": (110, 155)},
            "EastAsia": {"lat": (20, 50), "lon": (100, 140)},
            "Africa": {"lat": (-35, 35), "lon": (-20, 50)},
        }

        self.target_depth_m = target_depth_m
        
    def normalize_longitude_and_sort(self, ds, lon_name='lon'):
        """
        Normalize longitude in the dataset to [-180, 180] and sort by longitude.
    
        Parameters
        ----------
        ds : xr.DataArray or xr.Dataset
            Input data with a longitude coordinate.
        lon_name : str, default='lon'
            Name of the longitude coordinate.
    
        Returns
        -------
        xr.DataArray or xr.Dataset
            Dataset with normalized and sorted longitude.
        """
        if lon_name in ds.coords:
            lon = ds[lon_name]
            if lon.max() > 180:
                print("[INFO] Normalizing longitude to [-180, 180].")
                ds[lon_name] = ((lon + 180) % 360) - 180
                ds = ds.sortby(lon_name)
        return ds
    
    def compute_layer_integrated_sm(self, h2osoi, dzsoi, target_depth):
        if target_depth is not None: 
            dz_cumsum = dzsoi.cumsum(dim='levgrnd')
            mask = dz_cumsum <= target_depth
            h2osoi_masked = h2osoi.where(mask, 0.0)
            dzsoi_masked = dzsoi.where(mask, 0.0)
            sm = (h2osoi_masked * dzsoi_masked).sum(dim="levgrnd")
        else:
            sm = (h2osoi * dzsoi).sum(dim="levgrnd")
        sm = sm * 1000.0
        sm.attrs["units"] = "kg/m2"
        # Remove singleton lev dimension if present
        if 'levgrnd' in sm.dims and sm.sizes['levgrnd'] == 1:
            sm = sm.squeeze(dim='levgrnd')
        return sm

    def _resample_if_needed(self, ds):
        if self.expected_frequency.lower() == '6h':
            print("[INFO] Resampling 6-hourly data to daily means...")
            ds_daily = ds.resample(time='1D').mean()
            
            # Normalize time to start-of-day (00:00)
            ds_daily['time'] = pd.to_datetime(ds_daily['time'].values).normalize()
            return ds_daily
        return ds

    def _load_variable(self, varname, ensemble):
        original_var = varname
        possible_vars = [varname]
        if varname == "LHFLX":
            possible_vars = ["LHFLX", "QFLX"]

        if varname in ["SOILWATER_10CM", "H2OSOI"]:
            data_dir = self.data_dir.replace('atm', 'lnd')
        else:
            data_dir = self.data_dir

        alt_dir = data_dir.replace("daily", "6hourly")

        for v in possible_vars:
            for base_dir in [data_dir, alt_dir]:
                files = [os.path.join(base_dir, f"{v}.{ensemble}.{year}.nc") for year in self.years]
                if all(os.path.exists(f) for f in files):
                    if base_dir == alt_dir:
                        self.expected_frequency = '6h'
                        print(f"[INFO] {v} found in 6-hourly path: resampling will be applied.")

                    if original_var == "H2OSOI":
                        h2osoi = xr.concat([xr.open_dataset(f)["H2OSOI"] for f in files], dim="time")
                        h2osoi = self._resample_if_needed(h2osoi)
                        dzsoi = self.DZSOI 
                        sm = self.compute_layer_integrated_sm(h2osoi, dzsoi, self.target_depth_m)
                        if self.target_depth_m:
                            sm.name = f"SM_{int(self.target_depth_m * 100)}cm"
                        else:
                            sm.name = "SM_total"
                        ds = sm
                    else:
                        datasets = [xr.open_dataset(f)[v] for f in files]
                        ds = xr.concat(datasets, dim='time')
                        ds = self._resample_if_needed(ds)
                            
                    coord_renames = {}
                    for coord in ds.coords:
                        if coord.lower() == 'longitude':
                            coord_renames[coord] = 'lon'
                        elif coord.lower() == 'latitude':
                            coord_renames[coord] = 'lat'
                    if coord_renames:
                        print(f"[INFO] Renaming coordinates: {coord_renames}")
                        ds = ds.rename(coord_renames)
                        
                    ds = self.normalize_longitude_and_sort(ds)
                            
                    if self.landmask is not None and 'lat' in ds.coords and 'lon' in ds.coords:
                        try:
                            print("[INFO] Applying landmask...")
                            mask_interp = self.landmask.interp_like(ds, method='nearest')
                            ds = ds.where(mask_interp > 0.5)
                        except Exception as e:
                            print(f"[WARN] Failed to apply landmask: {e}")

                    if original_var == "LHFLX" and v == "QFLX":
                        print("[INFO] Converting QFLX to LHFLX using Lv.")
                        ds = ds * self.Lv
                        ds.attrs["units"] = "W/m2"
                        ds.name = "LHFLX"
                            
                    if self.lat_bounds:
                        ds = ds.sel(lat=slice(*self.lat_bounds))
                    if self.lon_bounds:
                        ds = ds.sel(lon=slice(*self.lon_bounds))

                    return ds

        raise FileNotFoundError(f"[ERROR] Could not find {possible_vars} in {data_dir} or {alt_dir} for {ensemble}.")
        
    def _compute_tci_dirmeyer(self, sm, flux):
        # Align by common time steps
        common_time = np.intersect1d(sm['time'], flux['time'])
        if len(common_time) < 2:
            raise ValueError("[ERROR] Not enough overlapping time points for regression.")
        
        sm = sm.sel(time=common_time)
        flux = flux.sel(time=common_time)
    
        # Stack spatial dims
        sm_stack = sm.stack(grid=('lat', 'lon')).transpose('time', 'grid')
        flux_stack = flux.stack(grid=('lat', 'lon')).transpose('time', 'grid')
    
        tci_vals = []
        for i in range(sm_stack.shape[1]):
            x = sm_stack[:, i].values
            y = flux_stack[:, i].values
            if (
                np.all(np.isfinite(x)) and
                np.all(np.isfinite(y)) and
                x.size > 1 and
                not np.allclose(x, x[0])
            ):
                slope, _, _, _, _ = linregress(x, y)
                sigma = np.std(x)
                tci_vals.append(slope * sigma)
            else:
                tci_vals.append(np.nan)
    
        tci_array = np.array(tci_vals).reshape((sm.sizes['lat'], sm.sizes['lon']))
        return xr.DataArray(
            tci_array, coords={'lat': sm.lat, 'lon': sm.lon},
            dims=['lat', 'lon'], name='TCI'
        )

    def compute_weighted_mean(self, tci_all):
        weights = np.cos(np.deg2rad(tci_all['lat']))
        weights.name = "weights"
        return tci_all.weighted(weights).mean(dim=['lat', 'lon'], skipna=True)

    def save_regional_mean(self, tci_all, tag, ens):
        data_vars = {}
    
        for region, bounds in self.regions.items():
            if bounds['lon'][0] < bounds['lon'][1]:
                tci_region = tci_all.sel(
                    lat=slice(*bounds['lat']),
                    lon=slice(*bounds['lon'])
                )
            else:
                west = tci_all.sel(lon=slice(bounds['lon'][0], 180))
                east = tci_all.sel(lon=slice(-180, bounds['lon'][1]))
                tci_region = xr.concat([west, east], dim='lon')
    
            mean_tci = self.compute_weighted_mean(tci_region)
            data_vars[f"TCI_{region}"] = mean_tci
    
        # Create a dataset with all regional means
        ds_out = xr.Dataset(data_vars)
    
        # Construct sub-identifier
        if self.sm_var == 'H2OSOI':
            if self.target_depth_m:
                sub = f"{self.sm_var}_{int(self.target_depth_m * 100)}cm_{self.flux_var}"
            else:
                sub = f"{self.sm_var}_alllayer_{self.flux_var}"
        else:
            sub = f"{self.sm_var}_{self.flux_var}"
    
        # Save one file with all regions
        outfile = os.path.join(self.outdir, f"TCI_window_{tag}_{ens}_{sub}_regional_mean.nc")
        if not os.path.exists(outfile):
            ds_out.to_netcdf(outfile)
            print(f"[SAVED] All regional means: {outfile}")

    def process(self, windows_dict, compute_regional_mean=False):
                    
        for ens in self.ensemble_list:
            print(f"[INFO] Checking outputs for {self.experiment} | {ens}")
    
            # Determine sub-tag for output filenames
            if self.sm_var == "H2OSOI" and self.target_depth_m:
                sub_tag = f"{self.sm_var}_{int(self.target_depth_m * 100)}cm_{self.flux_var}"
            else:
                sub_tag = f"{self.sm_var}_{self.flux_var}"
    
            # Filter windows that have not been processed
            windows_to_process = {}
            for tag, (start_str, end_str) in windows_dict.items():
                file_out = os.path.join(self.outdir, f"TCI_window_{tag}_{ens}_{sub_tag}.nc")
                if not os.path.exists(file_out):
                    windows_to_process[tag] = (start_str, end_str)
                else:
                    print(f"[SKIP] TCI already exists for window {tag} | {ens}")
    
            if not windows_to_process:
                print(f"[SKIP] All windows already processed for {ens}")
                continue
    
            print(f"[INFO] Processing TCI for {ens}, {len(windows_to_process)} windows need computation.")

            sm = self._load_variable(self.sm_var, ens)
            flux = self._load_variable(self.flux_var, ens)

            if not np.issubdtype(sm['time'].dtype, np.datetime64):
                base_time = np.datetime64(f"{self.years[0]}-01-01")
                sm['time'] = base_time + xr.to_timedelta(sm['time'], unit='h')
                flux['time'] = sm['time']

            for tag, (start_str, end_str) in windows_to_process.items():
                start = np.datetime64(start_str)
                end = np.datetime64(end_str)

                sm_window = sm.sel(time=slice(start, end))
                flux_window = flux.sel(time=slice(start, end))

                if sm_window.time.size < 2:
                    print(f"[WARN] Not enough data for window {tag}")
                    continue

                tci = self._compute_tci_dirmeyer(sm_window, flux_window)
                tci['time'] = sm_window.time.mean()
                tci = tci.expand_dims('time')


                file_out = os.path.join(self.outdir, f"TCI_window_{tag}_{ens}_{sub_tag}.nc")
                tci.to_netcdf(file_out)
                print(f"[SAVED] {file_out}")

                if compute_regional_mean:
                    self.save_regional_mean(tci, tag, ens)


In [9]:
if __name__ == "__main__":
    top_path = "/pscratch/sd/z/zhan391/e3sm_dart"
    out_path = "/pscratch/sd/z/zhan391/e3sm_dart/diag_dart"
    fig_path = "/people/zhan391/e3sm_dart_work/analysis/diagnostic/figures"
    landmask_file = "/pscratch/sd/z/zhan391/e3sm_dart/diag_dart/landmask_1x1.nc"
    soilayer_file = "/pscratch/sd/z/zhan391/e3sm_dart/diag_dart/dzsoi_elm.nc"

    exp_key = {
        "v3_spinup": 2011,
        "v3_hindcast": 2012
    }
    exp = 'v3_hindcast'
    exp_dict = get_experiment_dict(exp)
    years = [exp_key[exp]]

    # Labeled windows for TCI
    windows_dict = {
        "D1-15":   ("2012-01-01", "2012-01-15"),
        "D16-30":  ("2012-01-16", "2012-01-30"),
        "D31-45":  ("2012-01-31", "2012-02-14"),
        "D46-60":  ("2012-02-15", "2012-03-01"),
        "2012-01":  ("2012-01-01", "2012-01-31"),
        "2012-02":  ("2012-02-01", "2012-02-29")
    }
    
    for exp_name, exp_meta in exp_dict.items():
        print(f"\n[INFO] Running TCI for experiment: {exp_name}")
        path = exp_meta['path']
        template_str = exp_meta['template']
        template = Template(template_str.replace('%(', '${').replace(')s', '}'))
        nens = exp_meta['nens']
        ensemble_list = [f"EN{n:02d}" for n in range(1, nens + 1)]

        tci_calc = TerrestrialCouplingIndexCalculator(
            data_dir=path,
            experiment=exp_name,
            ensemble_list=ensemble_list,
            years=years,
            sm_var='H2OSOI',
            flux_var='LHFLX',
            target_depth_m=0.05,
            outdir=out_path,
            landmask_file=landmask_file,
            soilayer_file=soilayer_file
        )

        tci_calc.process(windows_dict=windows_dict, compute_regional_mean=True)



[INFO] Running TCI for experiment: CTRLEN10
[INFO] Normalizing longitude to [-180, 180].
[INFO] Checking outputs for CTRLEN10 | EN01
[SKIP] TCI already exists for window D1-15 | EN01
[SKIP] TCI already exists for window D16-30 | EN01
[SKIP] TCI already exists for window D31-45 | EN01
[SKIP] TCI already exists for window D46-60 | EN01
[SKIP] TCI already exists for window 2012-01 | EN01
[SKIP] TCI already exists for window 2012-02 | EN01
[SKIP] All windows already processed for EN01
[INFO] Checking outputs for CTRLEN10 | EN02
[SKIP] TCI already exists for window D1-15 | EN02
[SKIP] TCI already exists for window D16-30 | EN02
[SKIP] TCI already exists for window D31-45 | EN02
[SKIP] TCI already exists for window D46-60 | EN02
[SKIP] TCI already exists for window 2012-01 | EN02
[SKIP] TCI already exists for window 2012-02 | EN02
[SKIP] All windows already processed for EN02
[INFO] Checking outputs for CTRLEN10 | EN03
[SKIP] TCI already exists for window D1-15 | EN03
[SKIP] TCI already exi